In [11]:
from pathlib import Path
from pydantic import BaseModel, Field
import os

from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate


In [12]:
MODEL = "claude-sonnet-4-5-20250929"
DB_NAME = "../../vector_db_pro"
collection_name = "docs"
KNOWLEDGE_BASE_PATH = Path('../../knowledge-base-pro')

os.makedirs(DB_NAME, exist_ok=True)

In [13]:
def fetch_documents():
    documents = []

    loader = DirectoryLoader(KNOWLEDGE_BASE_PATH, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = 'wiki'
        documents.append(doc)

    print(f"Loaded {len(documents)} documents")
    return documents

In [14]:
documents = fetch_documents()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=250)
chunks = text_splitter.split_documents(documents)

Loaded 1 documents


In [15]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

def create_embeddings(chunks):
    if os.path.exists(DB_NAME):
        Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()
    
    ids = [str(i) for i in range(len(chunks))]
    Chroma.from_documents(ids=ids, documents=chunks, embedding=embeddings, persist_directory=DB_NAME)

create_embeddings(chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
retriever = vectorstore.as_retriever()
llm = ChatAnthropic(temperature=0, model_name=MODEL)

In [18]:

class RankingeSchema(BaseModel):
    ranking: list = Field(description="rankings")

def rerank(user_question, chunks):

    parser = PydanticOutputParser(pydantic_object=RankingeSchema)

    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked Return array no any other text e.g. {{"ranking": [1,2,3,4]}}.
"""

    user_prompt = "The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", user_prompt)
    ])

    formatted = prompt.format_messages(
        question=user_question,
        format_instructions=parser.get_format_instructions()
    )

    for msg in formatted:
        print(f"{msg.type.upper()}:\n{msg.content}\n{'-'*50}")

    chain = prompt | llm | parser

    response = chain.invoke({
        "question": user_question,
        "format_instructions": parser.get_format_instructions()
    })

    order = response.ranking
    print(order)
    return [chunks[i - 1] for i in order]


In [19]:
def fetch_context_unranked(question):
    return retriever.invoke(question)

def fetch_context_ranked(question, unranked_chunks):
    return rerank(question, unranked_chunks)

In [20]:
question = "who threw a bomb inside assembly?"
unranked_chunks = fetch_context_unranked(question);
print(fetch_context_ranked(question, unranked_chunks))

SYSTEM:

You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked Return array no any other text e.g. {"ranking": [1,2,3,4]}.

--------------------------------------------------
HUMAN:
The user has asked the following question:

who threw a bomb inside assembly?

Order all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.

Here are the chunks:

# CHUNK ID: 1:

violence and radical philosophy revived in the 1930s, when revolutionaries of
the Samiti a